# 🏥 Oxford Clinical AI Hackathon – Clinical AI Assistant

**Helping doctors with conversations, frameworks, and admin.**

This notebook demonstrates three core capabilities:

| Module | What it does |
|---|---|
| **Conversations** | Clinical chatbot, differential diagnosis, patient explanations, treatment options |
| **Frameworks** | SOAP notes, SBAR handover, NEWS2 calculator, discharge summaries, referral letters |
| **Admin** | Clinic letters, medication lists, MDT summaries, ICD-10 coding, task reminders |

> **Note:** Set `OPENAI_API_KEY` in the *Secrets* panel (🔑) for live AI responses.  
> Without a key, the app runs in **demo mode** with pre-written example outputs.


## 1 · Setup

In [ ]:
# Install dependencies
!pip install openai rich python-dotenv --quiet

In [ ]:
import os, sys

# ---- Optional: load API key from Colab Secrets ----
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✓ API key loaded from Colab Secrets")
except Exception:
    print("⚠ No API key found – running in DEMO MODE")

# ---- Make sure repo root is on the path ----
# If running from the repo root, this is a no-op.
repo_root = os.path.abspath(".")  # adjust if notebook is in a sub-folder
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
from utils.llm_client import LLMClient
from modules.conversation import ConversationAssistant
from modules.frameworks import ClinicalFrameworks
from modules.admin import AdminAssistant

llm = LLMClient()          # picks up OPENAI_API_KEY automatically
conv = ConversationAssistant(llm)
fw   = ClinicalFrameworks(llm)
adm  = AdminAssistant(llm)

mode = "DEMO" if llm.demo_mode else "LIVE (OpenAI)"
print(f"Clinical AI Assistant ready – {mode} mode")

---
## 2 · Conversations

### 2a · Clinical chatbot

In [ ]:
# Multi-turn clinical chatbot
conv.reset()  # start fresh

questions = [
    "What are the key diagnostic criteria for heart failure with reduced ejection fraction?",
    "Which diuretic would you typically start first?",
    "What are the target doses for an ACE inhibitor in this condition?",
]

for q in questions:
    print(f"\n>>> {q}")
    reply = conv.chat(q)
    print(reply)

### 2b · Differential diagnosis

In [ ]:
result = conv.differential_diagnosis(
    presenting_complaint="acute onset chest pain",
    age=58,
    sex="male",
    key_findings="diaphoretic, ST elevation in leads II III aVF, BP 90/60",
)
print(result)

### 2c · Patient explanation (plain English)

In [ ]:
medical_text = (
    "You have been diagnosed with type 2 diabetes mellitus. "
    "Your HbA1c is 72 mmol/mol, indicating suboptimal glycaemic control. "
    "We recommend initiating metformin 500 mg BD and arranging a retinal "
    "screening appointment to assess for diabetic retinopathy."
)

plain = conv.explain_for_patient(medical_text, reading_age=12)
print(plain)

---
## 3 · Clinical Frameworks

### 3a · SOAP note

In [ ]:
soap = fw.soap_note(
    subjective=(
        "68-year-old male presenting with 2-day history of increasing shortness "
        "of breath, productive cough with green sputum, fever, and right-sided "
        "pleuritic chest pain. Background of COPD (GOLD II) and type 2 diabetes."
    ),
    objective=(
        "Temp 38.6°C, HR 108, RR 24, SpO2 92% on air, BP 118/76. "
        "Right lower zone dullness to percussion, reduced air entry, "
        "coarse crackles. CXR: right lower lobe consolidation. "
        "WBC 14.2, CRP 187, Lactate 1.4."
    ),
    # Leave assessment and plan blank to let AI suggest them
)
print(soap)

### 3b · SBAR handover

In [ ]:
sbar = fw.sbar_handover(
    situation="Patient in bay 3 has become acutely confused and hypotensive in the last 20 minutes.",
    background="82-year-old female admitted yesterday with a UTI; background of hypertension and AF on warfarin.",
    assessment="NEWS2 score 7 – RR 22, HR 118, BP 88/52, Temp 38.9°C, SpO2 94%.",
    # recommendation left blank for AI to suggest
)
print(sbar)

### 3c · NEWS2 calculator

In [ ]:
result = fw.news2_score(
    respiratory_rate=22,
    spo2=94,
    systolic_bp=88,
    heart_rate=118,
    temperature=38.9,
    consciousness="C",         # New confusion
    on_supplemental_o2=False,
)

print(f"NEWS2 total  : {result['total']}")
print(f"Risk level   : {result['risk']}")
print(f"Recommendation: {result['recommendation']}")
print("\nBreakdown:")
for param, score in result["breakdown"].items():
    print(f"  {param:<22} {score}")

### 3d · Referral letter

In [ ]:
letter = fw.referral_letter(
    referring_doctor="Dr A Smith, Core Medical Trainee",
    receiving_specialty="Cardiology",
    patient_name="Mary Jones",
    dob="14/03/1961",
    nhs_number="987 654 3210",
    reason="Newly diagnosed heart failure with reduced ejection fraction (EF 35%), requesting specialist review and optimisation.",
    history=(
        "6-week history of increasing dyspnoea on exertion and ankle oedema. "
        "Echocardiogram: dilated LV, EF 35%, no significant valve disease."
    ),
    investigations="Echo as above; BNP 890 pg/mL; chest X-ray: cardiomegaly.",
    urgency="urgent",
)
print(letter)

---
## 4 · Admin

### 4a · Clinic letter

In [ ]:
clinic_letter = adm.clinic_letter(
    author="Dr B Taylor, Consultant Gastroenterologist",
    recipient="Dr C Brown, General Practitioner",
    patient_name="Robert Green",
    dob="22/11/1975",
    nhs_number="111 222 3333",
    clinic_date="09/04/2026",
    content_notes=(
        "Reviewed for ongoing abdominal pain. Flexible sigmoidoscopy last month: "
        "moderate active UC to the splenic flexure. Symptoms improving on "
        "mesalazine 2.4 g OD. Plan to continue current treatment, calprotectin "
        "in 3 months, review in 6 months."
    ),
)
print(clinic_letter)

### 4b · MDT summary

In [ ]:
mdt = adm.mdt_summary(
    patient_name="Patricia White, 71F",
    diagnosis="Stage IIIb non-small cell lung cancer (adenocarcinoma, EGFR +ve)",
    discussion_points=(
        "EGFR mutation confirmed (exon 19 deletion). "
        "CT shows right hilar mass, mediastinal LN involvement, no distant mets. "
        "Fit for systemic treatment (PS 1). "
        "Consensus: first-line osimertinib. Refer to clinical trials."
    ),
    attendees="Oncology, Respiratory, Radiology, Pathology, CNS",
)
print(mdt)

### 4c · ICD-10 code suggestions

In [ ]:
codes = adm.suggest_icd10_codes(
    "Patient with Type 2 diabetes with diabetic nephropathy, "
    "currently on haemodialysis for end-stage renal disease."
)
print(codes)

### 4d · Summarise clinical notes

In [ ]:
long_notes = """
10/04/2026 – Dr J Singh, CT2
Patient reviewed on ward round. Overnight no new events. Obs stable – BP 126/78,
HR 76, Sats 97% on air, Temp 36.8, RR 14. Reported slight improvement in 
dyspnoea compared to admission. Continues on IV co-amoxiclav and oral 
doxycycline for community-acquired pneumonia. Repeat CXR today shows partial 
resolution of right lower lobe consolidation. CRP down from 187 to 110. 
WBC 11.8 (down from 14.2). Renal function stable. Patient eating and drinking 
well; tolerating oral medications. Plan: step down to oral co-amoxiclav, 
continue doxycycline, repeat bloods tomorrow, anticipate discharge in 24–48h 
if continues to improve. Discussed with patient and family.
"""

summary = adm.summarise_notes(long_notes, max_words=80)
print(summary)

---
## 5 · Interactive CLI (optional)

If you prefer an interactive menu-driven interface, run the following cell.  
It starts the full CLI application inside the notebook.

> **Note:** The CLI is designed for terminal use. In Colab, input is provided via  
> the text prompt that appears below each cell when executed.

In [ ]:
# Uncomment the next line to launch the interactive CLI
# !python app.py